# Harmony best - mouse_mammary

Seeds: `0-9`

Metrics: BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR


In [1]:
import gc
import os
import sys

os.environ["PYTHONNOUSERSITE"] = "1"
sys.path = [
    path
    for path in sys.path
    if ".local/lib/python3.10/site-packages" not in path
]

PROJECT_ROOT = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics'
os.environ["R_HOME"] = f"{PROJECT_ROOT}/CondaEnv/harmony_optuna/lib/R"
sys.path.insert(0, f"{PROJECT_ROOT}/scib/")

import anndata as ad
import harmonypy as hm
import numpy as np
import pandas as pd
import scanpy as sc

from scib.metrics.metrics import metrics


In [2]:
DATASET_NAME = 'mouse_mammary'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEY = 'BATCH'
LABEL_KEY = 'celltype'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

N_SEEDS = 10
N_PCS = 50
N_NEIGHBORS = 15
N_TOP_HVG = 2000

EMBED_KEY = "X_harmony"
CLUSTER_KEY = "leiden_opt"
METRIC_BATCH_KEY = "harmony_metric_batch"
METRIC_LABEL_KEY = "harmony_metric_label"

assert DATASET_NAME == 'mouse_mammary'
assert BATCH_KEY == 'BATCH'
assert LABEL_KEY == 'celltype'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'BRAS', 'KBET', 'CPCR')

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_key": BATCH_KEY,
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(range(N_SEEDS)),
    },
    name="configuration",
)


dataset                                          mouse_mammary
data_path    /ubc/cs/research/beaver/projects/Single-cell-i...
batch_key                                                BATCH
label_key                                             celltype
metrics               [BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR]
seeds                           [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Name: configuration, dtype: object

In [3]:
def validate_dataset_contents():
    adata_backed = ad.read_h5ad(DATA_PATH, backed="r")
    try:
        required_keys = (BATCH_KEY, LABEL_KEY)
        missing_keys = [
            key for key in required_keys if key not in adata_backed.obs
        ]
        if missing_keys:
            raise ValueError(
                f"Missing obs keys {missing_keys}. "
                f"Available keys: {list(adata_backed.obs.columns)}"
            )

        obs = adata_backed.obs[[BATCH_KEY, LABEL_KEY]].copy()
        pseudotime = None
        if "TRAJ" in SELECTED_METRICS:
            if "dpt_pseudotime" not in adata_backed.obs:
                raise ValueError(
                    "TRAJ requires adata.obs['dpt_pseudotime']."
                )
            pseudotime = adata_backed.obs["dpt_pseudotime"].copy()

        n_obs = int(adata_backed.n_obs)
        n_vars = int(adata_backed.n_vars)
    finally:
        adata_backed.file.close()

    if obs.isna().any().any():
        raise ValueError(
            "Missing batch/label values: "
            f"{obs.isna().sum().to_dict()}"
        )

    n_batches = int(obs[BATCH_KEY].nunique())
    n_labels = int(obs[LABEL_KEY].nunique())
    if n_batches < 2:
        raise ValueError(
            f"Batch metrics require at least 2 batches; found {n_batches}."
        )

    batch_coverage = obs.groupby(
        LABEL_KEY,
        observed=True,
    )[BATCH_KEY].nunique()
    label_counts = obs.groupby(LABEL_KEY, observed=True).size()

    if "TRAJ" in SELECTED_METRICS:
        finite_pseudotime = pseudotime.dropna()
        if (
            len(finite_pseudotime) < 2
            or finite_pseudotime.nunique() < 2
        ):
            raise ValueError(
                "TRAJ requires at least 2 usable pseudotime values."
            )

    if "ISOLISI" in SELECTED_METRICS:
        min_batch_coverage = int(batch_coverage.min())
        if min_batch_coverage >= n_batches:
            raise ValueError(
                "ISOLISI is enabled, but every label appears in all batches."
            )

    eligible_cpcr_labels = [
        str(label)
        for label in label_counts.index
        if int(label_counts.loc[label]) >= 100
        and int(batch_coverage.loc[label]) >= 2
    ]
    if "CPCR" in SELECTED_METRICS and not eligible_cpcr_labels:
        raise ValueError(
            "CPCR requires a label with at least 100 cells across "
            "at least 2 batches."
        )

    return {
        "n_obs": n_obs,
        "n_vars": n_vars,
        "n_batches": n_batches,
        "n_labels": n_labels,
        "cpcr_eligible_labels": len(eligible_cpcr_labels),
    }


validate_dataset_contents()


{'n_obs': 9288,
 'n_vars': 1222,
 'n_batches': 3,
 'n_labels': 3,
 'cpcr_eligible_labels': 3}

In [4]:
OPTUNA_STUDY_NAME = 'harmony_mouse_mammary_joint_expanded_metrics_scgraphm_v2'
OPTUNA_BEST_TRIAL = 211
OPTUNA_BEST_VALUE = 0.7408612866966899
OPTUNA_BEST_PARAMS = {'theta': 7.865984153384493, 'sigma': 0.036031182713343934, 'lamb_mode': 'fixed', 'lamb': 0.11607743221159703, 'tau': 1.387709132344425, 'nclust': 193}

if OPTUNA_BEST_PARAMS["lamb_mode"] == "auto":
    resolved_lamb = -1
else:
    resolved_lamb = OPTUNA_BEST_PARAMS["lamb"]

HARMONY_BEST_ARGS = {
    "theta": OPTUNA_BEST_PARAMS["theta"],
    "lamb": resolved_lamb,
    "sigma": OPTUNA_BEST_PARAMS["sigma"],
    "nclust": OPTUNA_BEST_PARAMS["nclust"],
    "tau": OPTUNA_BEST_PARAMS["tau"],
    "block_size": 0.05,
    "max_iter_harmony": 10,
    "max_iter_kmeans": 20,
    "epsilon_cluster": 1e-5,
    "epsilon_harmony": 1e-4,
    "alpha": 0.2,
}


def run_harmony_best(adata, seed):
    adata_int = adata.copy()

    if "highly_variable" in adata_int.var:
        adata_int = adata_int[:, adata_int.var["highly_variable"]].copy()
    else:
        sc.pp.highly_variable_genes(
            adata_int,
            flavor="seurat",
            n_top_genes=N_TOP_HVG,
        )
        adata_int = adata_int[:, adata_int.var["highly_variable"]].copy()

    sc.tl.pca(
        adata_int,
        n_comps=N_PCS,
        svd_solver="arpack",
    )
    x_pca = np.asarray(adata_int.obsm["X_pca"], dtype=np.float32)

    harmony_result = hm.run_harmony(
        x_pca,
        adata_int.obs,
        BATCH_KEY,
        **HARMONY_BEST_ARGS,
        verbose=False,
        random_state=seed,
        device="cpu",
    )

    x_harmony = np.asarray(harmony_result.Z_corr)
    if (
        x_harmony.shape[0] != adata_int.n_obs
        and x_harmony.shape[1] == adata_int.n_obs
    ):
        x_harmony = x_harmony.T
    if x_harmony.shape[0] != adata_int.n_obs:
        raise ValueError(
            f"Harmony output shape {x_harmony.shape} is incompatible "
            f"with n_obs={adata_int.n_obs}."
        )
    if not np.all(np.isfinite(x_harmony)):
        raise ValueError("Harmony produced NaN or Inf in X_harmony.")

    adata_int.obsm[EMBED_KEY] = x_harmony
    return adata_int


pd.Series(
    {
        "study": OPTUNA_STUDY_NAME,
        "best_trial": OPTUNA_BEST_TRIAL,
        "best_value": OPTUNA_BEST_VALUE,
        **HARMONY_BEST_ARGS,
    },
    name="best_setting",
)


study               harmony_mouse_mammary_joint_expanded_metrics_s...
best_trial                                                        211
best_value                                                   0.740861
theta                                                        7.865984
lamb                                                         0.116077
sigma                                                        0.036031
nclust                                                            193
tau                                                          1.387709
block_size                                                       0.05
max_iter_harmony                                                   10
max_iter_kmeans                                                    20
epsilon_cluster                                               0.00001
epsilon_harmony                                                0.0001
alpha                                                             0.2
Name: best_setting, 

In [5]:
def metric_value(metric_series, key):
    if key not in metric_series.index:
        raise KeyError(
            f"Metric {key} not found. "
            f"Available metrics: {list(metric_series.index)}"
        )

    value = metric_series[key]
    if isinstance(value, (list, tuple, np.ndarray)):
        values = np.asarray(value, dtype=float)
        if values.size == 0 or not np.all(np.isfinite(values)):
            raise ValueError(
                f"Metric {key} returned non-finite values: {values}"
            )
        return float(np.mean(values))

    value = float(value)
    if not np.isfinite(value):
        raise ValueError(
            f"Metric {key} returned a non-finite value: {value}"
        )
    return value


def compute_joint_metrics(adata_raw, adata_int):
    sc.pp.neighbors(
        adata_int,
        use_rep=EMBED_KEY,
        n_neighbors=N_NEIGHBORS,
        random_state=0,
    )

    metric_key_map = {
        METRIC_BATCH_KEY: BATCH_KEY,
        METRIC_LABEL_KEY: LABEL_KEY,
    }
    for adata_name, adata_obj in (
        ("adata_raw", adata_raw),
        ("adata_int", adata_int),
    ):
        existing_aliases = [
            alias for alias in metric_key_map if alias in adata_obj.obs
        ]
        if existing_aliases:
            raise ValueError(
                f"Reserved metric obs keys already exist in {adata_name}: "
                f"{existing_aliases}"
            )
        for alias, source_key in metric_key_map.items():
            adata_obj.obs[alias] = adata_obj.obs[source_key].copy()

    metric_df = metrics(
        adata=adata_raw,
        adata_int=adata_int,
        batch_key=METRIC_BATCH_KEY,
        label_key=METRIC_LABEL_KEY,
        embed=EMBED_KEY,
        cluster_key=CLUSTER_KEY,
        verbose=False,
        bami_="BAMI" in SELECTED_METRICS,
        casw_="CASW" in SELECTED_METRICS,
        scgraphm_="SCGRAPHM" in SELECTED_METRICS,
        traj_="TRAJ" in SELECTED_METRICS,
        isolisi_="ISOLISI" in SELECTED_METRICS,
        bras_="BRAS" in SELECTED_METRICS,
        kbet_="KBET" in SELECTED_METRICS,
        cpcr_="CPCR" in SELECTED_METRICS,
    )
    metric_series = metric_df.iloc[:, 0]

    result = {
        key: metric_value(metric_series, key)
        for key in SELECTED_METRICS
    }
    bio_score = float(
        np.mean([result[key] for key in SELECTED_BIO_METRICS])
    )
    batch_score = float(
        np.mean([result[key] for key in SELECTED_BATCH_METRICS])
    )
    joint_score = float(0.6 * bio_score + 0.4 * batch_score)

    if not np.all(np.isfinite([bio_score, batch_score, joint_score])):
        raise ValueError(
            "Aggregation produced a non-finite score: "
            f"bio={bio_score}, batch={batch_score}, "
            f"joint={joint_score}"
        )

    result["bio_score"] = bio_score
    result["batch_score"] = batch_score
    result["joint_score"] = joint_score
    return result


In [6]:
results = []

for seed in range(N_SEEDS):
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_harmony_best(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()

results_df = pd.DataFrame(results)
results_df



===== seed 0 =====


2026-07-23 11:03:43,903 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...


2026-07-23 11:03:44,153 - harmonypy - INFO - KMeans initialization complete.


Cell type silhouette score...


Per cell type PC regression...


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.5713429226752323


optimised clustering against harmony_metric_label
optimal cluster resolution: 0.6
optimal score: 0.9579792802576879
BRAS...


seed           9.000000
BAMI           0.957979
CASW           0.658303
SCGRAPHM       0.718106
BRAS           0.949727
KBET           0.097713
CPCR           0.998299
bio_score      0.778129
batch_score    0.681913
joint_score    0.739643
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,0,0.967687,0.658212,0.718138,0.950214,0.091073,0.998457,0.781346,0.679915,0.740773
1,1,0.955487,0.658591,0.718179,0.949722,0.089610,0.998508,0.777419,0.679280,0.738164
2,2,0.961699,0.658335,0.718122,0.949838,0.093574,0.998382,0.779385,0.680598,0.739870
3,3,0.959174,0.658469,0.718175,0.949024,0.094994,0.998485,0.778606,0.680835,0.739497
4,4,0.960437,0.658298,0.718147,0.949132,0.093952,0.998359,0.778960,0.680481,0.739569
5,5,0.957680,0.658327,0.718147,0.949458,0.090536,0.998241,0.778051,0.679412,0.738595
6,6,0.959240,0.658202,0.718121,0.948827,0.089529,0.998065,0.778521,0.678807,0.738635
7,7,0.959811,0.658285,0.718115,0.949632,0.093778,0.998469,0.778737,0.680626,0.739493
8,8,0.959492,0.658337,0.718141,0.949305,0.099075,0.998377,0.778657,0.682253,0.740095
9,9,0.957979,0.658303,0.718106,0.949727,0.097713,0.998299,0.778129,0.681913,0.739643


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.959869,0.003056,0.955487,0.967687
CASW,0.658336,0.000110,0.658202,0.658591
SCGRAPHM,0.718139,0.000023,0.718106,0.718179
BRAS,0.949488,0.000400,0.948827,0.950214
KBET,0.093383,0.003111,0.089529,0.099075
CPCR,0.998364,0.000128,0.998065,0.998508
bio_score,0.778781,0.000995,0.777419,0.781346
batch_score,0.680412,0.001049,0.678807,0.682253
joint_score,0.739433,0.000738,0.738164,0.740773
